# Experiment 3: fitness-dependent movement

In [1]:
import os
from pathlib import Path

from AUTOclui import AUTOCommands as ac
from AUTOclui import runAUTO as ra
from pyvirtualdisplay import Display

In [2]:
folder = Path.cwd()
os.chdir(folder)

model_name = 'common_model'
output_folder = folder / 'output_experiment_three_fitness_dependent_movement'
output_folder.mkdir(exist_ok=True)

parameter_file = folder / 'experiment_parameters.dat'
parameter_file.write_text('0.2 0.02\n')

9

In [3]:
display = Display(visible=False, size=(1200, 900))
display.start()

In [4]:
runner = ra.runAUTO()

try:
    eq_forward = ac.run(e=model_name, c=model_name, runner=runner, NMX=4000, NPR=200)
    eq_backward = ac.run(DS='-', runner=runner, NMX=4000, NPR=200)
    eq = (eq_forward + eq_backward).relabel()
    ac.save(eq, 'eq')

    bp_curve = ac.run(
        eq('BP1'),
        ICP=[28, 31],
        ISW=2,
        DS=1.0e-3,
        DSMIN=1.0e-5,
        DSMAX=5.0e-3,
        NMX=8000,
        NPR=400,
        UZSTOP={28: [-0.25, 0.5], 31: [0.0, 4.0]},
        runner=runner,
    ).relabel()
    ac.save(bp_curve, 'bp_curve')

    try:
        lp_start = eq('LP1')
    except KeyError:
        lp_curve = None
        print('No LP1 label found in the equilibrium continuation; saving codim2 from BP curve only.')
    else:
        lp_curve = ac.run(
            lp_start,
            ICP=[28, 31],
            ISW=2,
            DS=1.0e-3,
            DSMIN=1.0e-5,
            DSMAX=5.0e-3,
            NMX=8000,
            NPR=400,
            UZSTOP={28: [-0.25, 0.5], 31: [0.0, 4.0]},
            runner=runner,
        ).relabel()
        ac.save(lp_curve, 'lp_curve')

    codim2 = bp_curve if lp_curve is None else (bp_curve + lp_curve).relabel()
    ac.save(codim2, 'codim2')
finally:
    runner.config(clean=True)
    ac.clean()


gfortran -g -fopenmp -O -c common_model.f90 -o common_model.o
gfortran -g -fopenmp -O common_model.o -o common_model.exe /auto/lib/*.o
Starting common_model ...

  BR    PT  TY  LAB       mu         L2-NORM          PL            FL            JL            PP            FP            JP      
   1     1  EP    1   0.00000E+00   9.55510E+00   0.00000E+00   6.15190E+00   0.00000E+00   0.00000E+00   7.31124E+00   0.00000E+00
   1     9  BP    2   1.13599E-01   9.55510E+00   8.03724E-21   6.15190E+00   5.97242E-21   7.78729E-21   7.31124E+00   5.29457E-21
   1    17  UZ    3   5.00000E-01   9.55510E+00  -1.00176E-32   6.15190E+00   1.72571E-32   8.51083E-32   7.31124E+00   5.09749E-32

  BR    PT  TY  LAB       mu         L2-NORM          PL            FL            JL            PP            FP            JP      
   2    48  LP    4   1.31405E-01   7.82963E+00   3.31970E-01   5.02185E+00   2.86496E-01   3.23271E-01   5.97674E+00   2.56489E-01
   2    68  MX    5   1.22216E-01   7.03669

In [5]:
p = ac.plot('eq', hide=True)
p.config(
    stability=True,
    grid=False,
    bifurcation_x=['mu'],
    bifurcation_y=['PL'],
    xlabel='mu',
    ylabel='PL',
    title='',
    minx=0.0,
    maxx=0.1,
)
p.savefig(str(output_folder / 'experiment_three_fitness_dependent_movement_1d.png'))
p.savefig(str(output_folder / 'experiment_three_fitness_dependent_movement_1d.svg'))

Created plot


In [6]:
p = ac.plot('codim2', hide=True)
p.config(
    grid=False,
    bifurcation_x=['beta'],
    bifurcation_y=['mu'],
    xlabel='beta',
    ylabel='mu',
    title='',
    minx=0.0,
    maxx=4.0,
    miny=0.0,
    maxy=0.1,
)
p.savefig(str(output_folder / 'experiment_three_fitness_dependent_movement_2d.png'))
p.savefig(str(output_folder / 'experiment_three_fitness_dependent_movement_2d.svg'))

Created plot


In [7]:
display.stop()
parameter_file.unlink(missing_ok=True)
for saved_name in ['eq', 'bp_curve', 'lp_curve', 'codim2']:
    try:
        ac.delete(saved_name)
    except Exception:
        pass


Deleting b.eq ... done
Deleting s.eq ... done
Deleting d.eq ... done
Deleting b.bp_curve ... done
Deleting s.bp_curve ... done
Deleting d.bp_curve ... done
Deleting b.lp_curve ... done
Deleting s.lp_curve ... done
Deleting d.lp_curve ... done
Deleting b.codim2 ... done
Deleting s.codim2 ... done
Deleting d.codim2 ... done
